# 02 - Périmètres d'agglomération

Ce notebook télécharge les communes d'agglomération depuis GeoAdmin et produit un périmètre dissous par agglomération.


In [1]:
# Paramètres
from pathlib import Path

PROJECT_ROOT = Path.cwd()
SOURCE = "vaco"
SWISS_ONLY = True
AGGLO_KEYS = ["lausanne", "berne", "geneve", "bale", "zurich"]
CRS_AGGLOMERATIONS = "EPSG:2056"

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Exécuter ce notebook depuis la racine du dépôt projet-slow-vaud.")


In [2]:
import sys

import geopandas as gpd
import pandas as pd

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from slowvaud.geoadmin import download_agglomeration_communes
from slowvaud.osm import dissolve_agglomeration_communes
from slowvaud.paths import ensure_data_tree, load_config


## Téléchargement GeoAdmin

La requête porte sur le nom exact configuré pour chaque agglomération. Si GeoAdmin ne renvoie rien, le notebook s'arrête.


In [3]:
cfg = load_config()
paths = ensure_data_tree()

summaries = []
for agglo_key in AGGLO_KEYS:
    output, metadata = download_agglomeration_communes(
        agglo_key,
        source=SOURCE,
        swiss_only=SWISS_ONLY,
        config=cfg,
    )
    summaries.append(metadata)

pd.DataFrame(summaries)


,agglomeration,source,layer,name_field,search_text,feature_count,swiss_only,source_label,download_url,notes,crs,output
0,lausanne,vaco,ch.are.agglomerationsverkehr,agglo_name,Lausanne,75,True,ARE - villes et agglomerations ayant droit aux...,https://data.geo.admin.ch/browser/index.html#/...,Basee sur la definition OFS de l'espace a cara...,EPSG:2056,/Users/marc-ed/BAS-local/repos/projet-slow-vau...
1,berne,vaco,ch.are.agglomerationsverkehr,agglo_name,Bern,49,True,ARE - villes et agglomerations ayant droit aux...,https://data.geo.admin.ch/browser/index.html#/...,Basee sur la definition OFS de l'espace a cara...,EPSG:2056,/Users/marc-ed/BAS-local/repos/projet-slow-vau...
2,geneve,vaco,ch.are.agglomerationsverkehr,agglo_name,Genève,92,True,ARE - villes et agglomerations ayant droit aux...,https://data.geo.admin.ch/browser/index.html#/...,Basee sur la definition OFS de l'espace a cara...,EPSG:2056,/Users/marc-ed/BAS-local/repos/projet-slow-vau...
3,bale,vaco,ch.are.agglomerationsverkehr,agglo_name,Basel,100,True,ARE - villes et agglomerations ayant droit aux...,https://data.geo.admin.ch/browser/index.html#/...,Basee sur la definition OFS de l'espace a cara...,EPSG:2056,/Users/marc-ed/BAS-local/repos/projet-slow-vau...
4,zurich,vaco,ch.are.agglomerationsverkehr,agglo_name,Zürich,143,True,ARE - villes et agglomerations ayant droit aux...,https://data.geo.admin.ch/browser/index.html#/...,Basee sur la definition OFS de l'espace a cara...,EPSG:2056,/Users/marc-ed/BAS-local/repos/projet-slow-vau...


## Contrôle CRS des fichiers bruts


In [4]:
raw_checks = []
for metadata in summaries:
    path = Path(metadata["output"])
    gdf = gpd.read_file(path)
    crs = str(gdf.crs)
    if crs != CRS_AGGLOMERATIONS:
        raise ValueError(f"CRS inattendu pour {path}: {crs}")
    raw_checks.append({
        "agglomeration": metadata["agglomeration"],
        "features": len(gdf),
        "crs": crs,
        "bounds": tuple(round(value, 2) for value in gdf.total_bounds),
    })

pd.DataFrame(raw_checks)


,agglomeration,features,crs,bounds
0,lausanne,75,EPSG:2056,"(2512518.0, 1145803.0, 2551159.0, 1177123.0)"
1,berne,49,EPSG:2056,"(2581820.0, 1181565.0, 2619062.0, 1219106.0)"
2,geneve,92,EPSG:2056,"(2485424.0, 1109578.0, 2518655.0, 1155690.0)"
3,bale,100,EPSG:2056,"(2595250.0, 1246272.0, 2639000.0, 1272264.0)"
4,zurich,143,EPSG:2056,"(2664527.0, 1222925.0, 2716149.0, 1276539.9)"


## Dissolution des périmètres


In [5]:
dissolved_rows = []
for metadata in summaries:
    agglo_key = metadata["agglomeration"]
    raw_path = Path(metadata["output"])
    output_path = paths["processed"] / "agglomerations" / SOURCE / f"{agglo_key}_dissolved.geojson"
    dissolve_agglomeration_communes(raw_path, output_path)

    dissolved = gpd.read_file(output_path)
    crs = str(dissolved.crs)
    if crs != CRS_AGGLOMERATIONS:
        raise ValueError(f"CRS inattendu pour {output_path}: {crs}")
    dissolved_rows.append({
        "agglomeration": agglo_key,
        "features": len(dissolved),
        "crs": crs,
        "output": str(output_path),
    })

pd.DataFrame(dissolved_rows)


,agglomeration,features,crs,output
0,lausanne,1,EPSG:2056,/Users/marc-ed/BAS-local/repos/projet-slow-vau...
1,berne,1,EPSG:2056,/Users/marc-ed/BAS-local/repos/projet-slow-vau...
2,geneve,1,EPSG:2056,/Users/marc-ed/BAS-local/repos/projet-slow-vau...
3,bale,1,EPSG:2056,/Users/marc-ed/BAS-local/repos/projet-slow-vau...
4,zurich,1,EPSG:2056,/Users/marc-ed/BAS-local/repos/projet-slow-vau...
